In [0]:
%pip install xgboost mlflow scikit-learn
dbutils.library.restartPython()

In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
from xgboost import XGBClassifier

from mlflow.models.signature import infer_signature

# Important for Unity Catalog
mlflow.set_registry_uri("databricks-uc")

df = spark.table("ws1.default.gold_train").toPandas()

X = df.drop(columns=["Response", "id"])
y = df["Response"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

mlflow.set_experiment("/Shared/vehicle-insurance")

with mlflow.start_run():

    model = XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        eval_metric="logloss"
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_val)
    prob = model.predict_proba(X_val)[:,1]

    acc = accuracy_score(y_val, pred)
    auc = roc_auc_score(y_val, prob)

    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc", auc)

    # signature required
    signature = infer_signature(X_train, model.predict(X_train))

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        signature=signature,
        input_example=X_train.head(5),
        registered_model_name="ws1.default.vehicle_insurance_model"
    )

    print("Accuracy:", acc)
    print("AUC:", auc)